In [ ]:
# This cell loads the original Superstore CSV file into a pandas DataFrame.
# It prints the raw row/column counts and shows the first few records so we can
# understand the initial structure of the data before any cleaning.
import pandas as pd
import numpy as np

# 1. Load the original raw file
raw_path = "superstore.csv"  
df = pd.read_csv(raw_path, encoding="ISO-8859-1")

# 2. Quick shape & preview
print("Raw shape (rows, columns):", df.shape)
df.head()

Raw shape (rows, columns): (9994, 29)


In [ ]:
# Overview of columns and dtypes
df.info()

# Quick descriptive stats for key numeric columns (including Quantity & Discount)
df[["Sales", "Quantity", "Discount", "Profit"]].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Row ID           9994 non-null   int64  
 1   Order ID         9994 non-null   object 
 2   Order Date       9994 non-null   object 
 3   Ship Date        9994 non-null   object 
 4   Ship Mode        9994 non-null   object 
 5   Customer ID      9994 non-null   object 
 6   Customer Name    9994 non-null   object 
 7   Customer_no      9994 non-null   int64  
 8   Segment          9994 non-null   object 
 9   Segment_no       9994 non-null   int64  
 10  Country          9994 non-null   object 
 11  City             9994 non-null   object 
 12  State            9994 non-null   object 
 13  State_no         9994 non-null   int64  
 14  Postal Code      9994 non-null   int64  
 15  Region           9994 non-null   object 
 16  Region_no        9994 non-null   int64  
 17  Product ID    

In [ ]:
# 3.1 Convert date columns to datetime
date_cols = ["Order Date", "Ship Date"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# 3.2 Ensure key numeric columns are numeric types
numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Check again after conversions
df[["Order Date", "Ship Date"] + numeric_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Order Date  9994 non-null   datetime64[ns]
 1   Ship Date   9994 non-null   datetime64[ns]
 2   Sales       9994 non-null   float64       
 3   Quantity    9994 non-null   int64         
 4   Discount    9994 non-null   float64       
 5   Profit      9994 non-null   float64       
dtypes: datetime64[ns](2), float64(3), int64(1)
memory usage: 468.6 KB
/tmp/ipykernel_116/972903534.py:6: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")
/tmp/ipykernel_116/972903534.py:6: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this war

In [ ]:
# 4.1 Basic distribution of Quantity
print(df["Quantity"].describe())

# 4.2 Look at the most extreme quantities
print("\nTop 10 largest Quantity values:")
print(df["Quantity"].sort_values(ascending=False).head(10))

# 4.3 How many rows are above a reasonable threshold?
# From earlier experience, genuine values are mostly between 1 and ~15.
qty_threshold = 100  # very conservative; will capture obvious errors like 10,000

high_qty = df[df["Quantity"] > qty_threshold]
print(f"\nNumber of rows with Quantity > {qty_threshold}:", high_qty.shape[0])
high_qty[["Order ID", "Product Name", "Quantity", "Sales", "Profit"]].head()

count     9994.000000
mean         6.789674
std        173.188374
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max      10000.000000
Name: Quantity, dtype: float64

Top 10 largest Quantity values:
4846    10000
8649    10000
5337    10000
968        14
4206       14
3158       14
4440       14
203        14
6914       14
4969       14
Name: Quantity, dtype: int64

Number of rows with Quantity > 100: 3


In [ ]:
# 5.1 Flag quantity outliers
qty_threshold = 100  # keep same as above
df["Quantity_Outlier"] = df["Quantity"] > qty_threshold

print("Quantity outlier counts:")
print(df["Quantity_Outlier"].value_counts())

# 5.2 Create a cleaned dataframe without quantity outliers
df_clean = df[~df["Quantity_Outlier"]].copy()

print("\nShape after removing quantity outliers:", df_clean.shape)

Quantity outlier counts:
Quantity_Outlier
False    9991
True        3
Name: count, dtype: int64

Shape after removing quantity outliers: (9991, 30)


In [ ]:
# 6.1 Basic distribution of Discount
print(df_clean["Discount"].describe())

# 6.2 List any rows with impossible discount values
discount_bad = df_clean[(df_clean["Discount"] < 0) | (df_clean["Discount"] > 1)]
print("\nRows with Discount < 0 or > 1:")
print(discount_bad[["Order ID", "Product Name", "Discount", "Sales", "Profit"]].head())
print("Total bad discount rows:", discount_bad.shape[0])

count    9991.000000
mean        0.156410
std         0.207051
min         0.000000
25%         0.000000
50%         0.200000
75%         0.200000
max         1.300000
Name: Discount, dtype: float64

Rows with Discount < 0 or > 1:
            Order ID                      Product Name  Discount    Sales  \
846   CA-2014-167850            AT&T 1070 Corded Phone       1.2  178.384   
6396  CA-2016-149272  Novimex Swivel Fabric Task Chair       1.3  528.430   

       Profit  
846    22.298  
6396 -143.431  
Total bad discount rows: 2


In [ ]:
# 7.1 Flag discount outliers
df_clean["Discount_Outlier"] = (df_clean["Discount"] < 0) | (df_clean["Discount"] > 1)

print("Discount outlier counts:")
print(df_clean["Discount_Outlier"].value_counts())

# 7.2 Filter them out
df_clean = df_clean[~df_clean["Discount_Outlier"]].copy()

print("\nShape after removing discount outliers:", df_clean.shape)

Discount outlier counts:
Discount_Outlier
False    9989
True        2
Name: count, dtype: int64

Shape after removing discount outliers: (9989, 31)


In [ ]:
# This cell checks the cleaned dataset to confirm that Quantity and Discount are now valid.
# It prints summary statistics and counts any remaining extreme or impossible values
# (e.g. Quantity > 100 or Discount outside the 0–1 range) as a final data-quality check.
print("Final Quantity stats (cleaned):")
print(df_clean["Quantity"].describe())

print("\nAny Quantity > 100 remaining?")
print(df_clean[df_clean["Quantity"] > 100].shape[0])

print("\nFinal Discount stats (cleaned):")
print(df_clean["Discount"].describe())

print("\nAny Discount < 0 or > 1 remaining?")
print(df_clean[(df_clean["Discount"] < 0) | (df_clean["Discount"] > 1)].shape[0])

Final Quantity stats (cleaned):
count    9989.000000
mean        3.789068
std         2.225039
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        14.000000
Name: Quantity, dtype: float64

Any Quantity > 100 remaining?
0

Final Discount stats (cleaned):
count    9989.000000
mean        0.156191
std         0.206491
min         0.000000
25%         0.000000
50%         0.200000
75%         0.200000
max         0.800000
Name: Discount, dtype: float64

Any Discount < 0 or > 1 remaining?
0


In [ ]:
# Drop the outlier flags from the saved file if you don't want them in future analysis
cols_to_drop = ["Quantity_Outlier", "Discount_Outlier"]
for col in cols_to_drop:
    if col in df_clean.columns:
        df_clean = df_clean.drop(columns=[col])

# Save cleaned dataset
clean_path = "Superstore_cleaned.csv"
df_clean.to_csv(clean_path, index=False)

print(f"Cleaned dataset saved to: {clean_path}")
print("Final cleaned shape:", df_clean.shape)

Cleaned dataset saved to: Superstore_cleaned.csv
Final cleaned shape: (9989, 29)


In [ ]:
# This cell reloads the already-cleaned Superstore dataset that was saved to disk.
# It prints the shape and previews the first few rows as a sanity check before
# using this cleaned version for all further analysis and modelling.
import pandas as pd

df = pd.read_csv("Superstore_cleaned.csv")

print("Loaded cleaned dataset. Shape:", df.shape)
df.head()

Loaded cleaned dataset. Shape: (9989, 29)


In [ ]:
# Overview of columns and types
df.info()

# Quick look at unique values per column
unique_counts = df.nunique().sort_values(ascending=False)
print("\nUnique value count per column:\n")
print(unique_counts)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9989 entries, 0 to 9988
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Row ID           9989 non-null   int64  
 1   Order ID         9989 non-null   object 
 2   Order Date       9989 non-null   object 
 3   Ship Date        9989 non-null   object 
 4   Ship Mode        9989 non-null   object 
 5   Customer ID      9989 non-null   object 
 6   Customer Name    9989 non-null   object 
 7   Customer_no      9989 non-null   int64  
 8   Segment          9989 non-null   object 
 9   Segment_no       9989 non-null   int64  
 10  Country          9989 non-null   object 
 11  City             9989 non-null   object 
 12  State            9989 non-null   object 
 13  State_no         9989 non-null   int64  
 14  Postal Code      9989 non-null   int64  
 15  Region           9989 non-null   object 
 16  Region_no        9989 non-null   int64  
 17  Product ID    

In [ ]:
# Missing values per column
missing = df.isna().sum().sort_values(ascending=False)
print("\nMissing values per column:\n")
print(missing)


Missing values per column:

Row ID             0
Order ID           0
Order Date         0
Ship Date          0
Ship Mode          0
Customer ID        0
Customer Name      0
Customer_no        0
Segment            0
Segment_no         0
Country            0
City               0
State              0
State_no           0
Postal Code        0
Region             0
Region_no          0
Product ID         0
Category           0
Category_no        0
Sub-Category       0
Sub-Category_no    0
Product Name       0
Product Name_no    0
Sales              0
Quantity           0
Discount           0
Profit             0
Returned           0
dtype: int64


In [ ]:
# This cell defines the main numeric variables of interest and computes their
# descriptive statistics (count, mean, min, max, quartiles).
# It provides a quick overview of the central tendency and spread of Sales,
# Quantity, Discount and Profit.
numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]

print("\nDescriptive statistics for key numeric variables:\n")
print(df[numeric_cols].describe().T)


Descriptive statistics for key numeric variables:

           count        mean         std       min     25%     50%      75%  \
Sales     9989.0  229.781936  623.325495     0.444  17.248  54.384  209.940   
Quantity  9989.0    3.789068    2.225039     1.000   2.000   3.000    5.000   
Discount  9989.0    0.156191    0.206491     0.000   0.000   0.200    0.200   
Profit    9989.0   28.668866  234.310812 -6599.978   1.728   8.662   29.364   

                max  
Sales     22638.480  
Quantity     14.000  
Discount      0.800  
Profit     8399.976  


In [ ]:
# This cell automatically identifies all categorical (object-type) columns in the dataset.
# It lists their names and prints the top value counts for each, helping us understand
# the key categories (e.g. segments, regions, product types) and their frequency.
categorical_cols = df.select_dtypes(include="object").columns

print("\nCategorical columns:", list(categorical_cols))

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(10))

In [ ]:
# This cell loads the cleaned dataset and visualises the distribution of order-level profit
# using a histogram created with Plotly. It also prints detailed profit summary statistics
# and counts how many orders made a loss, giving an immediate view of profitability patterns.
import pandas as pd
import plotly.express as px

# Load data
data = pd.read_csv("Superstore_cleaned.csv")

# Plot histogram
fig = px.histogram(
    data,
    x="Profit",
    nbins=50,
    title="Distribution of Order-Level Profit",
    labels={"Profit": "Profit (£)"},
)
fig.show()

# Basic statistics
print(data["Profit"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print("\nLoss-making orders count:", (data["Profit"] < 0).sum())

count    9989.000000
mean       28.668866
std       234.310812
min     -6599.978000
1%       -319.317348
5%        -52.988640
50%         8.662000
95%       168.470400
99%       580.742512
max      8399.976000
Name: Profit, dtype: float64

Loss-making orders count: 1870


In [ ]:
# Checking for outliers in Profit and Losses
import pandas as pd
import numpy as np
import plotly.express as px

# Load the version you saved after cleaning quantity & discount
df = pd.read_csv("Superstore_cleaned.csv", parse_dates=["Order Date", "Ship Date"])

print(df[["Sales", "Quantity", "Discount", "Profit"]].describe())

              Sales     Quantity     Discount       Profit
count   9989.000000  9989.000000  9989.000000  9989.000000
mean     229.781936     3.789068     0.156191    28.668866
std      623.325495     2.225039     0.206491   234.310812
min        0.444000     1.000000     0.000000 -6599.978000
25%       17.248000     2.000000     0.000000     1.728000
50%       54.384000     3.000000     0.200000     8.662000
75%      209.940000     5.000000     0.200000    29.364000
max    22638.480000    14.000000     0.800000  8399.976000


In [ ]:
# Basic quantiles for profit
# These were real business scenerios and not outliers
quantiles = df["Profit"].quantile([0.001, 0.01, 0.05, 0.95, 0.99, 0.999])
print("Selected profit quantiles:\n", quantiles, "\n")

print("Min profit:", df["Profit"].min())
print("Max profit:", df["Profit"].max())

# Show the 10 largest losses and profits
print("\nMost negative profits:")
display(df.nsmallest(10, "Profit")[["Order ID", "Product Name", "Quantity", "Discount", "Sales", "Profit"]])

print("\nMost positive profits:")
display(df.nlargest(10, "Profit")[["Order ID", "Product Name", "Quantity", "Discount", "Sales", "Profit"]])

Selected profit quantiles:
 0.001   -1666.804514
0.010    -319.317348
0.050     -52.988640
0.950     168.470400
0.990     580.742512
0.999    2402.204771
Name: Profit, dtype: float64 

Min profit: -6599.978
Max profit: 8399.976

Most negative profits:



Most positive profits:


In [ ]:
# Having a Helicopter view of analysisng total sales, profit and loss
import pandas as pd
import plotly.express as px

# If needed, reload your cleaned file (adjust path/name if different)
df = pd.read_csv("Superstore_cleaned.csv")

# 1. Aggregate metrics
total_sales = df['Sales'].sum()
total_profit = df.loc[df['Profit'] > 0, 'Profit'].sum()
total_loss = df.loc[df['Profit'] < 0, 'Profit'].sum()   # this is negative
total_loss_abs = abs(total_loss)                        # for display

print("High-level profitability overview:\n")
print(f"Total Sales Revenue : £{total_sales:,.0f}")
print(f"Total Profit        : £{total_profit:,.0f}")
print(f"Total Loss          : £{total_loss:,.0f}")   # will show as negative
print(f"Total Loss (abs)    : £{total_loss_abs:,.0f}")

# 2. Create a simple bar chart (using absolute loss so bars are comparable)
summary_df = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Total Loss (absolute)"],
    "Amount": [total_sales, total_profit, total_loss_abs]
})

fig = px.bar(
    summary_df,
    x="Metric",
    y="Amount",
    title="Overall Financial Picture – Sales, Profit, and Loss",
    text="Amount"
)

fig.update_traces(texttemplate="£%{text:,.0f}", textposition="outside")
fig.update_yaxes(title="Amount (£)")
fig.update_layout(xaxis_title="", uniformtext_minsize=10, uniformtext_mode='hide')

fig.show()

High-level profitability overview:

Total Sales Revenue : £2,295,292
Total Profit        : £442,361
Total Loss          : £-155,988
Total Loss (abs)    : £155,988


In [ ]:
# Convert date columns to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

# Make a copy so we can experiment safely
data = df.copy()

# Shipping time in days
data["Ship_Time"] = (data["Ship Date"] - data["Order Date"]).dt.days

# Guard against divide-by-zero
data["Quantity"] = data["Quantity"].replace(0, np.nan)

# Unit price
data["Unit_Price"] = data["Sales"] / data["Quantity"]

# Profit margin
data["Profit_Margin"] = data["Profit"] / data["Sales"].replace(0, np.nan)

# Discount bands for easier visualisation
bins = [-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1.0]
labels = ["0%", "0–10%", "10–20%", "20–30%", "30–50%", "50%+"]

data["Discount_Band"] = pd.cut(data["Discount"], bins=bins, labels=labels)

# Time-based fields
data["Order_Year"] = data["Order Date"].dt.year
data["Order_Month"] = data["Order Date"].dt.to_period("M").astype(str)
data["Order_Quarter"] = data["Order Date"].dt.to_period("Q").astype(str)

data.head()

In [ ]:
# the large losses are real behaviour.
# Digging deeper into loss making transaction to know where business needs improvement
import pandas as pd

# 1. Build profit breakdown per Category
profit_breakdown = (
    df.groupby("Category")["Profit"]
      .agg(
          Positive_Profit=lambda x: x[x > 0].sum(),
          Loss=lambda x: x[x < 0].sum(),          # will be negative
          Net_Profit="sum"
      )
      .reset_index()
)

# 2. Add an absolute-loss column for easier reading
profit_breakdown["Loss_Abs"] = -profit_breakdown["Loss"]

# 3. Reorder columns for a cleaner view
profit_breakdown = profit_breakdown[
    ["Category", "Positive_Profit", "Loss", "Loss_Abs", "Net_Profit"]
]

profit_breakdown

In [ ]:
import plotly.express as px

# Use Loss_Abs so the bar height for loss is visually comparable
# this code generate profit breakdown by each category
plot_data = profit_breakdown.melt(
    id_vars="Category",
    value_vars=["Positive_Profit", "Loss_Abs", "Net_Profit"],
    var_name="Metric",
    value_name="Amount"
)

fig = px.bar(
    plot_data,
    x="Category",
    y="Amount",
    color="Metric",
    barmode="group",
    title="Profit Breakdown by Category – Positive Profit, Loss, and Net Profit",
    text_auto=".0f"
)

fig.update_yaxes(title_text="Amount (£)")
fig.update_xaxes(title_text="Category")
fig.update_traces(textposition="outside")
fig.show()

In [ ]:
# Overall loss (absolute value of all negative profits)
# Checking why total loss is not equal to 155,988 which was actual loss of company in sub-categories
total_loss = -df.loc[df["Profit"] < 0, "Profit"].sum()
total_profit = df["Profit"].sum()

print(f"Total profit (net): £{total_profit:,.0f}")
print(f"Total loss (sum of negative rows): £{total_loss:,.0f}")

Total profit (net): £286,373
Total loss (sum of negative rows): £155,988


In [ ]:
# Losses by sub-category, using only negative-profit rows
loss_by_subcat = (
    df.loc[df["Profit"] < 0]
      .groupby("Sub-Category")["Profit"]
      .sum()
      .sort_values()               # most negative at the bottom
)

loss_by_subcat_abs = -loss_by_subcat  # convert to positive numbers for readability

print("Loss by Sub-Category (only negative orders):")
print(loss_by_subcat_abs)

print("\nCheck sum of all sub-category losses:")
print(f"£{loss_by_subcat_abs.sum():,.0f}  (should match total_loss)")

Loss by Sub-Category (only negative orders):
Sub-Category
Binders        38510.4964
Tables         32412.1483
Machines       30118.6682
Bookcases      12152.2060
Chairs          9737.4103
Appliances      8629.6412
Phones          7530.6235
Furnishings     6490.9134
Storage         6426.3038
Supplies        3015.6219
Accessories      930.6265
Fasteners         33.1952
Name: Profit, dtype: float64

Check sum of all sub-category losses:
£155,988  (should match total_loss)


In [ ]:
# Net profit per sub-category (what you plotted earlier)
# loss inside sub-categories still made net profit overall, except for 3 weak sub-categories.
net_profit_by_subcat = (
    df.groupby("Sub-Category")["Profit"]
      .sum()
      .rename("Net_Profit")
)

# Pure losses per sub-category (only negative rows)
loss_only_by_subcat = (
    df.loc[df["Profit"] < 0]
      .groupby("Sub-Category")["Profit"]
      .sum()
      .rename("Loss_Only")
)

# Join them together
subcat_profit_summary = (
    pd.concat([net_profit_by_subcat, loss_only_by_subcat], axis=1)
      .fillna(0)
)

# Make the loss column positive for readability
subcat_profit_summary["Loss_Only"] = -subcat_profit_summary["Loss_Only"]

# Also add a "Positive_Profit_Only" column
subcat_profit_summary["Positive_Profit_Only"] = (
    subcat_profit_summary["Net_Profit"] + subcat_profit_summary["Loss_Only"]
)

subcat_profit_summary = subcat_profit_summary.sort_values("Net_Profit", ascending=True)

subcat_profit_summary

In [ ]:

# Filter all negative-profit transactions
df_clean = pd.read_csv("Superstore_cleaned.csv")
# Filter all negative-profit transactions
loss_df = df_clean[df_clean['Profit'] < 0].copy()

print("Total loss (should match previous £156k):", loss_df['Profit'].sum())
loss_df.head()

Total loss (should match previous £156k): -155987.85469999997


In [ ]:
# Loss breakdown by sub-category
loss_by_subcat = loss_df.groupby('Sub-Category')['Profit'].sum().sort_values()

loss_by_subcat

In [ ]:
#12 sub-category made loss transaction which now equals to 160k. 
import plotly.express as px

loss_plot = loss_by_subcat.reset_index().copy()
loss_plot["Loss"] = loss_plot["Profit"].abs()   # convert negative to positive

fig = px.bar(
    loss_plot,
    x="Sub-Category",
    y="Loss",
    title="Loss Breakdown by Sub-Category (Positive Bars)",
    labels={"Loss": "Total Loss (£)"},
)
fig.show()

In [ ]:
#Loss breakdown by product
loss_by_product = (
    loss_df.groupby(['Product Name'])['Profit']
    .sum()
    .sort_values()
)

loss_by_product.head(20)  # top 20 most loss-making products

In [ ]:
# Count of Loss-Making Transactions per Product
loss_counts = (
    loss_df.groupby('Product Name')
    .size()
    .sort_values(ascending=False)
)

loss_counts.head(20)

In [ ]:
# Total loss plus total loss transaction of top 20 products
loss_product_summary = (
    loss_df.groupby('Product Name')
    .agg(
        Total_Loss=('Profit', 'sum'),
        Loss_Transactions=('Profit', 'count')
    )
    .sort_values('Total_Loss')
)

loss_product_summary.head(20)

In [ ]:
# Analyse why these products lost money
df_clean['Implied_Cost'] = df_clean['Sales'] - df_clean['Profit']
df_clean['Unit_Price'] = df_clean['Sales'] / df_clean['Quantity']
df_clean['Unit_Cost'] = df_clean['Implied_Cost'] / df_clean['Quantity']
df_clean['Margin_Per_Unit'] = df_clean['Unit_Price'] - df_clean['Unit_Cost']

In [ ]:
import numpy as np
# Create a binary flag to classify each order as Profit or Loss,  
# then compare average discount levels across these two groups.  
# This helps diagnose whether heavy discounting is a major driver of loss-making transactions
df_clean['Profit_Flag'] = np.where(df_clean['Profit'] < 0, 'Loss', 'Profit')

discount_compare = (
    df_clean.groupby('Profit_Flag')['Discount']
    .agg(['mean', 'median'])
)

discount_compare

In [ ]:
# Group transactions by Profit/Loss and compare their average unit prices.  
# This identifies whether loss-making orders tend to involve different pricing levels  
# and helps uncover pricing-related risks.
unitprice_compare = (
    df_clean.groupby('Profit_Flag')['Unit_Price']
    .agg(['mean', 'median'])
)

unitprice_compare

In [ ]:
# Calculate mean and median margin per unit for Profit vs Loss groups.  
# This highlights structural margin weaknesses in loss-making orders  
# and supports identifying products that fail to achieve sustainable margins.
margin_compare = (
    df_clean.groupby('Profit_Flag')['Margin_Per_Unit']
    .agg(['mean', 'median'])
)

margin_compare

In [ ]:
# Plot a boxplot to visualise how discount rates differ between Profit and Loss orders.  
# This visual makes the relationship between discounting intensity and loss risk  
# immediately clear and supports discount-policy recommendations.
fig = px.box(
    df_clean,
    x="Profit_Flag",
    y="Discount",
    title="Discount Levels — Profit vs Loss Transactions",
    labels={"Discount": "Discount Rate"},
)

fig.show()

In [ ]:
# Plot a boxplot to visualise how unit prices differ between Profit and Loss orders.
fig = px.box(
    df_clean,
    x="Profit_Flag",
    y="Unit_Price",
    title="Unit Price — Profit vs Loss Transactions",
)

fig.show()
# Loss making transaction diagnosis done

In [ ]:
# Loss analysis done, now we will move towards strength of business, which is to analyse where profit comes
# want to see whether profit is dominated by a few extreme orders
df = pd.read_csv("Superstore_cleaned.csv")
fig = px.histogram(
    data,
    x="Profit",
    nbins=50,
    title="Distribution of Order-Level Profit",
    labels={"Profit": "Profit (£)"},
)
fig.show()

# Basic statistics
print(data["Profit"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print("\nLoss-making orders count:", (data["Profit"] < 0).sum())

count    9989.000000
mean       28.668866
std       234.310812
min     -6599.978000
1%       -319.317348
5%        -52.988640
50%         8.662000
95%       168.470400
99%       580.742512
max      8399.976000
Name: Profit, dtype: float64

Loss-making orders count: 1870


In [ ]:
# Total profit by category
cat_profit = (
    data.groupby("Category")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Profit", ascending=False)
)

fig = px.bar(
    cat_profit,
    x="Category",
    y="Profit",
    title="Total Profit by Category",
    labels={"Profit": "Total Profit (£)"},
    text_auto=".0f"
)
fig.update_traces(textposition="outside")
fig.show()

cat_profit

In [ ]:
# Total profit by subcategory
subcat_profit = (
    data.groupby("Sub-Category")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Profit", ascending=False)
)

fig = px.bar(
    subcat_profit,
    x="Sub-Category",
    y="Profit",
    title="Total Profit by Sub-Category",
    labels={"Profit": "Total Profit (£)"},
)
fig.update_layout(xaxis={"categoryorder": "total descending"})
fig.show()

subcat_profit.head(10), subcat_profit.tail(10)

In [ ]:
# Sales vs Profit by Sub-Category
subcat_sales_profit = (
    data.groupby("Sub-Category")[["Sales", "Profit"]]
    .sum()
    .reset_index()
)

fig = px.scatter(
    subcat_sales_profit,
    x="Sales",
    y="Profit",
    text="Sub-Category",
    title="Sub-Category: Sales vs Profit",
    labels={"Sales": "Total Sales (£)", "Profit": "Total Profit (£)"},
)
fig.update_traces(textposition="top center")
fig.show()

In [ ]:
# Aggregate Sales and Profit by Sub-Category
subcat = (
    data.groupby("Sub-Category")[["Sales", "Profit"]]
    .sum()
    .reset_index()
)

# Calculate thresholds
median_sales = subcat["Sales"].median()
median_profit = subcat["Profit"].median()
low_sales_cutoff = subcat["Sales"].quantile(0.25)

# Classification into 3 groups only
def classify(row):
    sales = row["Sales"]
    profit = row["Profit"]

    # 1️⃣ Low Sales or Negative Profit → Low Performance
    if sales <= low_sales_cutoff or profit < 0:
        return "❗ Low Sales / Low or Negative Profit"

    # 2️⃣ High Sales + High Profit → Top Performers
    if sales > median_sales and profit > median_profit:
        return "✔ High Sales + High Profit"

    # 3️⃣ Everything else that is profitable → Moderate Sales, Good Profit
    return "✔ Moderate Sales + Good Profit"

# Apply classifier
subcat["Category_Group"] = subcat.apply(classify, axis=1)

subcat

In [ ]:
# Create a scatter plot to compare total sales and total profit for each sub-category,  
# colouring points by performance group and labeling them for easy interpretation.
fig = px.scatter(
    subcat,
    x="Sales",
    y="Profit",
    color="Category_Group",
    text="Sub-Category",
    title="Sub-Category Performance Groups (3 Categories)",
    labels={"Sales": "Total Sales (£)", "Profit": "Total Profit (£)"}
)

fig.update_traces(textposition="top center")
fig.show()

In [ ]:
# Profit by customer segment
seg_profit = (
    data.groupby("Segment")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Profit", ascending=False)
)

fig = px.bar(
    seg_profit,
    x="Segment",
    y="Profit",
    title="Total Profit by Customer Segment",
    labels={"Profit": "Total Profit (£)"},
    text_auto=".0f",
)
fig.update_traces(textposition="outside")
fig.show()

seg_profit

In [ ]:
# Reveals which combinations of segment and category are strong vs weak.
seg_cat = (
    data.groupby(["Segment", "Category"])["Profit"]
    .sum()
    .reset_index()
)

fig = px.density_heatmap(
    seg_cat,
    x="Category",
    y="Segment",
    z="Profit",
    color_continuous_scale="RdBu",
    title="Profit Heatmap: Segment × Category",
    labels={"Profit": "Total Profit (£)"}
)
fig.show()

seg_cat

In [ ]:
# Profit by region
region_profit = (
    data.groupby("Region")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Profit", ascending=False)
)

fig = px.bar(
    region_profit,
    x="Region",
    y="Profit",
    title="Total Profit by Region",
    labels={"Profit": "Total Profit (£)"},
    text_auto=".0f",
)
fig.update_traces(textposition="outside")
fig.show()

region_profit

In [ ]:
import plotly.express as px

# Calculate total profit per state
state_profit = (
    data.groupby("State")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Profit", ascending=False)
)

# Split into top and bottom 10
top_states = state_profit.head(10)
bottom_states = state_profit.tail(10)

# --- Chart 1: Top 10 States by Profit ---
fig = px.bar(
    top_states,
    x="State",
    y="Profit",
    title="Top 10 States by Profit",
    labels={"Profit": "Total Profit (£)"}
)
fig.show()

# --- Chart 2: Bottom 10 States (Losses Shown as Upward Bars) ---
bottom_states_plot = bottom_states.copy()
bottom_states_plot["Loss (Absolute)"] = bottom_states_plot["Profit"].abs()

fig = px.bar(
    bottom_states_plot.sort_values("Profit"),
    x="State",
    y="Loss (Absolute)",
    title="Bottom 10 States by Loss (Shown as Upward Bars)",
    labels={"Loss (Absolute)": "Total Loss (£)"}
)
fig.show()

top_states, bottom_states

In [ ]:
# Provides context before linking shipping time to profit.
fig = px.histogram(
    data,
    x="Ship_Time",
    nbins=15,
    title="Distribution of Shipping Time (Days)",
    labels={"Ship_Time": "Shipping Time (days)"}
)
fig.show()

data["Ship_Time"].describe()

In [ ]:
# Shipping mode vs profit
shipmode_profit = (
    data.groupby("Ship Mode")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Profit", ascending=False)
)

fig = px.bar(
    shipmode_profit,
    x="Ship Mode",
    y="Profit",
    title="Total Profit by Shipping Mode",
    labels={"Profit": "Total Profit (£)"}
)
fig.show()

shipmode_profit

In [ ]:
!pip install statsmodels==0.14.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 92.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 85.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
# This code creates a scatter plot to examine whether shipping time has any linear
# relationship with profit, using an OLS trendline to visually assess correlation
# and understand whether longer delivery durations contribute to higher loss risk.
fig = px.scatter(
    data,
    x="Ship_Time",
    y="Profit",
    trendline="ols",
    title="Profit vs Shipping Time",
    labels={"Ship_Time": "Shipping Time (days)", "Profit": "Profit (£)"},
    opacity=0.5
)
fig.show()

In [ ]:
# setting context for discount vs profit analysis
fig = px.histogram(
    data,
    x="Discount",
    nbins=20,
    title="Distribution of Discounts",
    labels={"Discount": "Discount (decimal)"},
)
fig.show()

data["Discount"].describe()

In [ ]:
# This code groups all transactions by discount band and calculates total profit in each band.
disc_band_profit = (
    data.groupby("Discount_Band")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Discount_Band")
)

fig = px.bar(
    disc_band_profit,
    x="Discount_Band",
    y="Profit",
    title="Total Profit by Discount Band",
    labels={"Profit": "Total Profit (£)", "Discount_Band": "Discount Band"},
)
fig.show()

disc_band_profit

/tmp/ipykernel_116/1454134070.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [ ]:
# Checking relation of discount and qty sold
# Define discount bands
bins = [0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 1.0]
labels = ['0–10%', '10–20%', '20–30%', '30–40%', '40–50%', '50–60%', '60%+']

df_clean['Discount_Band'] = pd.cut(df_clean['Discount'], bins=bins, labels=labels, right=False)

# Summarise average sales and quantity per discount band
discount_summary = (
    df_clean
    .groupby('Discount_Band')
    .agg(
        Avg_Sales=('Sales', 'mean'),
        Avg_Quantity=('Quantity', 'mean'),
        Orders=('Sales', 'size')
    )
    .reset_index()
)

discount_summary

/tmp/ipykernel_116/2728085121.py:11: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [ ]:
# 3a) Bar chart – Average Sales per order by discount band
# Discount are not increasing qty sold means discount is not being properly placed
# business is giving away profit without much extra volume
fig = px.bar(
    discount_summary,
    x='Discount_Band',
    y='Avg_Sales',
    title='Average Sales per Line by Discount Band',
    text=discount_summary['Avg_Sales'].round(0)
)
fig.update_layout(xaxis_title='Discount Band', yaxis_title='Average Sales per Line (£)')
fig.show()

# 3b) Bar chart – Average Quantity per order by discount band
fig = px.bar(
    discount_summary,
    x='Discount_Band',
    y='Avg_Quantity',
    title='Average Quantity per Line by Discount Band',
    text=discount_summary['Avg_Quantity'].round(2)
)
fig.update_layout(xaxis_title='Discount Band', yaxis_title='Average Quantity per Line')
fig.show()

In [ ]:
# This code creates quantity bands (e.g., 1, 2, 3, 4–5, 6–10, 11+) so we can analyse
# how total profit varies across different order sizes.
qty_bins = [0, 1, 2, 3, 5, 10, data["Quantity"].max()]
qty_labels = ["1", "2", "3", "4–5", "6–10", "11+"]

data["Qty_Band"] = pd.cut(data["Quantity"], bins=qty_bins, labels=qty_labels)

qty_band_profit = (
    data.groupby("Qty_Band")["Profit"]
    .sum()
    .reset_index()
)

fig = px.bar(
    qty_band_profit,
    x="Qty_Band",
    y="Profit",
    title="Total Profit by Quantity Band",
    labels={"Qty_Band": "Quantity Band", "Profit": "Total Profit (£)"}
)
fig.show()

qty_band_profit

/tmp/ipykernel_116/591722975.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [ ]:
# Profit by year
# Ensure Order_Year is integer
data["Order_Year"] = data["Order Date"].dt.year.astype(int)

# Group by year and sum the profit
year_profit = (
    data.groupby("Order_Year")["Profit"]
    .sum()
    .reset_index()
)

# Plot
fig = px.bar(
    year_profit,
    x="Order_Year",
    y="Profit",
    title="Total Profit by Year",
    labels={"Order_Year": "Year", "Profit": "Total Profit (£)"}
)

fig.update_layout(xaxis=dict(type="category"))  # Treat years as categories
fig.show()

year_profit

In [ ]:
# Profit by month
month_profit = (
    data.groupby("Order_Month")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Order_Month")
)

fig = px.line(
    month_profit,
    x="Order_Month",
    y="Profit",
    title="Monthly Profit Trend",
    labels={"Order_Month": "Order Month", "Profit": "Total Profit (£)"}
)
fig.update_xaxes(tickangle=45)
fig.show()

month_profit.head()

In [ ]:
#Profit by quater
quarter_profit = (
    data.groupby("Order_Quarter")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Order_Quarter")
)

fig = px.bar(
    quarter_profit,
    x="Order_Quarter",
    y="Profit",
    title="Total Profit by Quarter",
    labels={"Order_Quarter": "Quarter", "Profit": "Total Profit (£)"}
)
fig.show()

quarter_profit

In [ ]:
product_profit = (
    data.groupby(["Product ID", "Product Name"])["Profit"]
    .sum()
    .reset_index()
    .sort_values("Profit", ascending=False)
)

product_profit["Cum_Profit"] = product_profit["Profit"].cumsum()
total_profit_prod = product_profit["Profit"].sum()
product_profit["Cum_Profit_Pct"] = product_profit["Cum_Profit"] / total_profit_prod
product_profit["Product_Rank_Pct"] = np.arange(1, len(product_profit) + 1) / len(product_profit)

fig = px.line(
    product_profit,
    x="Product_Rank_Pct",
    y="Cum_Profit_Pct",
    title="Product Pareto Curve – Cumulative Profit",
    labels={
        "Product_Rank_Pct": "Cumulative Share of Products",
        "Cum_Profit_Pct": "Cumulative Share of Profit"
    }
)
fig.add_hline(y=0.8, line_dash="dash", annotation_text="80% profit line", annotation_position="bottom right")
fig.show()

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Total profit by product
prod_profit = (
    data.groupby("Product Name")["Profit"]
    .sum()
    .reset_index()
)

# 2. Sort from most to least profitable
prod_profit = prod_profit.sort_values("Profit", ascending=False)

# 3. Compute share and cumulative share of total profit
total_profit = prod_profit["Profit"].sum()

prod_profit["Profit_Share"] = prod_profit["Profit"] / total_profit
prod_profit["Cum_Profit_Share"] = prod_profit["Profit_Share"].cumsum()

# 4. Keep only products that lie in the first 80% of cumulative profit
top80_products = prod_profit[prod_profit["Cum_Profit_Share"] <= 0.80].copy()

# See how many products and inspect a few
len(top80_products), top80_products.head(10)

In [ ]:
# Take top 15 products within the 80% group for a clear chart
top_plot = top80_products.head(15).copy()

# Convert profit share to percentage for nicer reading (optional)
top_plot["Profit_Share_Pct"] = top_plot["Profit_Share"] * 100

fig = px.bar(
    top_plot,
    x="Product Name",
    y="Profit_Share",
    title="Top Products Contributing to First 80% of Total Profit",
    labels={
        "Product Name": "Product",
        "Profit_Share": "Share of Total Profit"
    },
    text="Profit_Share_Pct"
)

# Format y-axis and text as %
fig.update_yaxes(tickformat=".0%")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(xaxis_tickangle=-45)

fig.show()

top_plot[["Product Name", "Profit", "Profit_Share_Pct"]]

In [ ]:
# Diagnostic, descriptive analysis done now Moving on to the predictive analysis
# creating new variables


import pandas as pd  # <-- REQUIRED

# creating new variables
df_clean = pd.read_csv("Superstore_cleaned.csv")

df_clean['Unit_Price'] = df_clean['Sales'] / df_clean['Quantity']

In [ ]:
df_clean['Margin_Per_Unit'] = df_clean['Profit'] / df_clean['Quantity']

In [ ]:
# Import all core libraries needed for the supervised modelling stage,
# including tools for splitting the data, building preprocessing pipelines,
# fitting Ridge and Random Forest regressors, and evaluating model performance.


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import plotly.express as px

In [ ]:
# Load the cleaned file you saved earlier
df_clean = pd.read_csv("Superstore_cleaned.csv")
df_clean['Unit_Price'] = df_clean['Sales'] / df_clean['Quantity']
df_clean['Margin_Per_Unit'] = df_clean['Profit'] / df_clean['Quantity']
df_clean.head()

In [ ]:
# ---- Dates to datetime ----
df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])
df_clean['Ship Date']  = pd.to_datetime(df_clean['Ship Date'])

# ---- Ship time in days ----
df_clean['Ship_Time'] = (df_clean['Ship Date'] - df_clean['Order Date']).dt.days

# ---- Time features ----
df_clean['Order_Year']    = df_clean['Order Date'].dt.year
df_clean['Order_Month']   = df_clean['Order Date'].dt.month
df_clean['Order_Quarter'] = df_clean['Order Date'].dt.quarter
df_clean['Order_DOW']     = df_clean['Order Date'].dt.dayofweek  # 0=Mon, 6=Sun

# ---- Unit price (sales per unit) ----
df_clean['Unit_Price'] = df_clean['Sales'] / df_clean['Quantity']

# (Margin_Per_Unit or anything calculated from Profit is NOT used as a feature –
# we only used it for diagnostics earlier.)

In [ ]:
# Quickly preview the main numerical variables used in the models
# to sanity-check their values and confirm that the cleaned dataset
# contains the expected columns before building any models.

df_clean[['Sales', 'Quantity', 'Discount', 'Profit', 'Ship_Time', 'Unit_Price']].head()

In [ ]:
# Define the modelling target (Profit), select business-relevant numeric
# and categorical features, and combine them into X. Then split the data
# into training and validation sets (80/20) to allow out-of-sample
# evaluation of all subsequent regression models.
y = df_clean['Profit']

# Choose explanatory features (business-meaningful and not ID codes)
numeric_features = [
    'Sales',
    'Quantity',
    'Discount',
    'Ship_Time',
    'Unit_Price',
    'Order_Month',
    'Order_Quarter',
    'Order_DOW'
]

categorical_features = [
    'Category',
    'Sub-Category',
    'Segment',
    'Region',
    'Ship Mode',
]

X = df_clean[numeric_features + categorical_features]

# Train / validation split (20% hold-out)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_valid.shape

In [ ]:
# This block builds the preprocessing pipeline for the models.
# Numeric features are standardised and categorical features are one-hot encoded,
# then both are combined into a single ColumnTransformer called `preprocessor`.

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Categorical: one-hot encode
categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessor ready.")

Preprocessor ready.


In [ ]:
# This block creates a full modelling pipeline using the shared preprocessor
# followed by a Ridge regression model, and then fits it on the training data.
# It prepares a baseline regression model to see how well profit can be predicted.
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=10.0, random_state=42))
])

ridge_pipeline.fit(X_train, y_train)

print("Ridge model fitted.")

Ridge model fitted.


In [ ]:
# This block defines a helper function to evaluate any regression model using
# R², MAE and RMSE on both train and validation sets, and then applies it
# to the Ridge pipeline so we can quantitatively assess its performance.

def evaluate_model(name, model, X_tr, y_tr, X_va, y_va):
    y_tr_pred = model.predict(X_tr)
    y_va_pred = model.predict(X_va)

    metrics = {
        'Model' : name,
        'Train_R2': r2_score(y_tr, y_tr_pred),
        'Valid_R2': r2_score(y_va, y_va_pred),
        'Valid_MAE': mean_absolute_error(y_va, y_va_pred),
        'Valid_RMSE': np.sqrt(mean_squared_error(y_va, y_va_pred))
    }
    return metrics

ridge_metrics = evaluate_model(
    "Ridge",
    ridge_pipeline,
    X_train, y_train,
    X_valid, y_valid
)
ridge_metrics

In [ ]:
# Get feature names after preprocessing
preprocess_fitted = ridge_pipeline.named_steps['preprocessor']

# Numeric names unchanged; categorical expanded to one-hot
feature_names = np.concatenate([
    preprocess_fitted.named_transformers_['num'].named_steps['scaler'].get_feature_names_out(numeric_features),
    preprocess_fitted.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features)
])

# Coefficients from Ridge model
coefs = ridge_pipeline.named_steps['model'].coef_

ridge_importance = (
    pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefs,
        'Abs_Coeff': np.abs(coefs)
    })
    .sort_values('Abs_Coeff', ascending=False)
)

ridge_importance.head(20)

In [ ]:
top_n = 15
fig = px.bar(
    ridge_importance.head(top_n),
    x='Feature',
    y='Coefficient',
    title=f"Top {top_n} Ridge Coefficients – Direction & Strength",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Build a Random Forest regression pipeline using the same preprocessing steps as Ridge.
# The model is configured with 300 trees and trained on the training set to predict order-level profit.
# This gives us a more flexible non-linear benchmark to compare against the simpler Ridge model.
rf_preprocessor = preprocessor  # reuse same definition

rf_pipeline = Pipeline(steps=[
    ('preprocessor', rf_preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

print("Random Forest fitted.")

Random Forest fitted.


In [ ]:
# Evaluate the fitted Random Forest model on both training and validation data.
# This cell calculates R², MAE and RMSE so we can directly compare Random Forest performance to Ridge.
# The printed metrics show whether the model is learning real signal or mainly overfitting noise.
rf_metrics = evaluate_model(
    "RandomForest",
    rf_pipeline,
    X_train, y_train,
    X_valid, y_valid
)
rf_metrics

In [ ]:
# Combine the stored Ridge and Random Forest metric dictionaries into a single DataFrame.
# This summary table makes it easy to compare training/validation R² and error values across models.
# It provides the evidence used later in the report to reject both regression approaches for forecasting.
import pandas as pd

pd.DataFrame([ridge_metrics, rf_metrics])

In [ ]:
# Feature names after preprocessing (same logic as before)
# Extract the feature names after preprocessing and match them to the Random Forest feature importances.
# This cell builds a sorted table showing which variables the Random Forest relied on most when predicting profit.
# Even though the overall model is weak, these importances help cross-check which factors appear most influential.
rf_preprocess_fitted = rf_pipeline.named_steps['preprocessor']

rf_feature_names = np.concatenate([
    rf_preprocess_fitted.named_transformers_['num'].named_steps['scaler'].get_feature_names_out(numeric_features),
    rf_preprocess_fitted.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features)
])

rf_importances = rf_pipeline.named_steps['model'].feature_importances_

rf_importance = (
    pd.DataFrame({
        'Feature': rf_feature_names,
        'Importance': rf_importances
    })
    .sort_values('Importance', ascending=False)
)

rf_importance.head(20)

In [ ]:
top_n = 15
fig = px.bar(
    rf_importance.head(top_n),
    x='Feature',
    y='Importance',
    title=f"Top {top_n} Random Forest Feature Importances",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# This block recreates all engineered date-based and transaction-level features 
# to ensure consistency after reloading the cleaned dataset. It converts date columns, 
# computes unit price and shipping time, and extracts calendar components for modelling analysis.
import pandas as pd
import numpy as np

# --- Ensure dates are datetime (in case notebook restarted) ---
df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])
df_clean['Ship Date']  = pd.to_datetime(df_clean['Ship Date'])

# --- Recreate engineered features if they are missing ---

# Unit price = sales per unit
df_clean['Unit_Price'] = df_clean['Sales'] / df_clean['Quantity']

# Shipping time in days
df_clean['Ship_Time'] = (df_clean['Ship Date'] - df_clean['Order Date']).dt.days

# Order calendar features
df_clean['Order_Year']    = df_clean['Order Date'].dt.year
df_clean['Order_Month']   = df_clean['Order Date'].dt.month
df_clean['Order_Quarter'] = df_clean['Order Date'].dt.quarter
df_clean['Order_DOW']     = df_clean['Order Date'].dt.dayofweek  # 0=Mon, 6=Sun

# Optional sanity check
print(df_clean[['Unit_Price', 'Ship_Time', 'Order_Year', 'Order_Month',
                'Order_Quarter', 'Order_DOW']].head())

   Unit_Price  Ship_Time  Order_Year  Order_Month  Order_Quarter  Order_DOW
0       2.968          3        2017           11              4          0
1       2.968          4        2017            9              3          6
2       3.710          6        2017           10              4          0
3       3.304          7        2015           11              4          1
4       4.130          5        2016            4              2          4


In [ ]:
# Both models have near-zero or negative R² on hold-out data, so we do not rely on them for forecasting.
# 1. Target: 1 = loss, 0 = profit / break-even
df_clean['Is_Loss'] = (df_clean['Profit'] < 0).astype(int)

# 2. Features: reuse the same engineered columns you already have
feature_cols = [
    'Sales', 'Discount', 'Quantity', 'Unit_Price',
    'Ship_Time', 'Order_Year', 'Order_Month', 'Order_Quarter', 'Order_DOW',
    'Category', 'Sub-Category', 'Segment', 'Region', 'Ship Mode'
]

X = df_clean[feature_cols].copy()
y = df_clean['Is_Loss']

# 3. Train / validation split
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 4. Preprocessor (numeric + categorical)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

num_features = ['Sales', 'Discount', 'Quantity', 'Unit_Price',
                'Ship_Time', 'Order_Year', 'Order_Month',
                'Order_Quarter', 'Order_DOW']

cat_features = ['Category', 'Sub-Category', 'Segment', 'Region', 'Ship Mode']

numeric_transformer = 'passthrough'
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ]
)

# 5. Logistic Regression pipeline
logit_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, solver='liblinear'))
])

logit_pipeline.fit(X_train, y_train)

In [ ]:
# This block evaluates the Logistic Regression model using accuracy, ROC AUC, 
# and a full classification report. It compares predicted probabilities with actual outcomes 
# to assess how well the model identifies loss-making vs profit-making orders.
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

y_valid_pred_proba = logit_pipeline.predict_proba(X_valid)[:, 1]
y_valid_pred = (y_valid_pred_proba >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_valid, y_valid_pred))
print("ROC AUC:", roc_auc_score(y_valid, y_valid_pred_proba))
print(classification_report(y_valid, y_valid_pred))

Accuracy: 0.944944944944945
ROC AUC: 0.9823671370324515
              precision    recall  f1-score   support

           0       0.94      0.99      0.97      2436
           1       0.97      0.73      0.83       561

    accuracy                           0.94      2997
   macro avg       0.95      0.86      0.90      2997
weighted avg       0.95      0.94      0.94      2997



In [ ]:
# This code extracts the learned logistic regression coefficients after preprocessing.
# It retrieves the one-hot encoded categorical feature names, combines them with numeric features,
# and creates a sorted table showing which features most strongly increase or decrease loss risk.
import numpy as np

logit_model = logit_pipeline.named_steps['model']
ohe = logit_pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(cat_features)

all_feature_names = num_features + list(cat_feature_names)
coefs = logit_model.coef_[0]

coef_df = (pd.DataFrame({'Feature': all_feature_names, 'Coefficient': coefs})
             .sort_values('Coefficient', ascending=False))

coef_df.head(15), coef_df.tail(15)

In [ ]:
import numpy as np
import pandas as pd

# Extract trained logistic regression model
logit_model = logit_pipeline.named_steps['model']

# Extract the one-hot encoder from the preprocessor
ohe = logit_pipeline.named_steps['preprocessor'].named_transformers_['cat']

# Get names of encoded categorical features
cat_feature_names = ohe.get_feature_names_out(cat_features)

# Combine numeric + categorical feature names
all_feature_names = num_features + list(cat_feature_names)

# Extract the learned coefficients
coefs = logit_model.coef_[0]

# Build coefficient dataframe
coef_df = (
    pd.DataFrame({
        'Feature': all_feature_names,
        'Coefficient': coefs
    })
)

coef_df.head()
import plotly.express as px

# How many features to show in each plot
top_n = 12

# --- 1. Features that INCREASE loss risk (positive coefficients) ---
coef_pos = (
    coef_df[coef_df["Coefficient"] > 0]
    .sort_values("Coefficient", ascending=False)
    .head(top_n)
)

fig_pos = px.bar(
    coef_pos,
    x="Coefficient",
    y="Feature",
    orientation="h",
    title="Logistic Regression – Top Features that Increase Loss Risk",
    labels={"Coefficient": "Effect on Loss Risk (log-odds)", "Feature": ""}
)

# Order features from weakest at bottom to strongest at top and adjust height
fig_pos.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=40 * len(coef_pos)  # scale height so nothing is cut off
)

fig_pos.show()


# --- 2. Features that REDUCE loss risk (negative coefficients) ---
coef_neg = (
    coef_df[coef_df["Coefficient"] < 0]
    .sort_values("Coefficient")  # most negative first
    .head(top_n)
)

fig_neg = px.bar(
    coef_neg,
    x="Coefficient",
    y="Feature",
    orientation="h",
    title="Logistic Regression – Top Features that Reduce Loss Risk",
    labels={"Coefficient": "Effect on Loss Risk (log-odds)", "Feature": ""}
)

fig_neg.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=40 * len(coef_neg)
)

fig_neg.show()

In [ ]:
# This code prepares a fresh copy of the cleaned dataset specifically for K-Means clustering.
# It reloads the cleaned file, creates a duplicate dataframe for safety, and computes key unit metrics
# (unit price and margin per unit) that will serve as input features for clustering analysis.

import numpy as np
import pandas as pd

# Work on a copy so we don't break anything earlier
df_clean = pd.read_csv('Superstore_cleaned.csv')   # <-- FIXED HERE
df_k = df_clean.copy()

# Basic unit metrics
df_k['Unit_Price'] = df_k['Sales'] / df_k['Quantity']
df_k['Margin_Per_Unit'] = df_k['Profit'] / df_k['Quantity']

In [ ]:
# This block prepares features for transaction-level clustering and scales them
# before fitting a 3-cluster K-means model. The resulting 'Trans_Cluster' labels
# segment orders into distinct patterns of sales, discounts, quantities and margin per unit.

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Features for transaction-level clustering
trans_features = ['Sales', 'Profit', 'Discount', 'Quantity', 'Margin_Per_Unit']

X_trans = df_k[trans_features].copy()

# Scale features so K-means isn't dominated by £ amounts
scaler_trans = StandardScaler()
X_trans_scaled = scaler_trans.fit_transform(X_trans)

# K-means with 3 clusters
kmeans_trans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_k['Trans_Cluster'] = kmeans_trans.fit_predict(X_trans_scaled)

In [ ]:
# This block summarises each transaction cluster by averaging key metrics
# (sales, profit, discount, quantity, margin per unit) and counting how many
# orders fall into each group, so we can interpret the financial profile of every cluster.
trans_profile = (
    df_k
    .groupby('Trans_Cluster')[trans_features]
    .mean()
    .round(2)
)

trans_profile['Count'] = df_k.groupby('Trans_Cluster').size()
trans_profile

In [ ]:
# Make a copy so we can work with it
summary = trans_profile.copy()

# Cluster with highest average discount and lowest profit → "High Discount – High Loss"
high_disc_loss_idx = (
    summary
    .assign(
        disc_rank=summary['Discount'].rank(ascending=False),
        profit_rank=summary['Profit'].rank(ascending=True)  # lowest profit = rank 1
    )
    .sort_values(['disc_rank', 'profit_rank'])
    .index[0]
)

# Cluster with highest sales and positive profit → "High Sales – Healthy Profit"
high_sales_idx = summary['Sales'].idxmax()

labels_trans = {}
labels_trans[high_disc_loss_idx] = 'High Discount – High Loss'
labels_trans[high_sales_idx] = 'High Sales – Healthy Profit'

# Any remaining cluster → "Low Sales – Low Risk"
for idx in summary.index:
    if idx not in labels_trans:
        labels_trans[idx] = 'Low Sales – Low Risk'

# Attach labels
df_k['Trans_Cluster_Label'] = df_k['Trans_Cluster'].map(labels_trans)
trans_profile['Label'] = trans_profile.index.map(labels_trans)

trans_profile

In [ ]:
# This block extracts the learned logistic regression coefficients and matches them
# to their corresponding feature names after one-hot encoding, allowing us to
# interpret which variables most strongly push a transaction toward loss or profit.
import plotly.express as px

# Sample to keep the plot light
sample = df_k.sample(2000, random_state=42)

px.scatter(
    sample,
    x='Discount',
    y='Profit',
    color='Trans_Cluster_Label',
    title='K-means Transaction Clusters: Discount vs Profit',
    hover_data=['Sales', 'Quantity', 'Category', 'Sub-Category']
)

In [ ]:
# This block aggregates sales, profit, discount, and units at Sub-Category level
# to compute total performance metrics for each product group.
# It also derives unit price and margin per unit for deeper profitability comparison.
sub_agg = (
    df_k
    .groupby('Sub-Category')
    .agg(
        Total_Sales=('Sales', 'sum'),
        Total_Profit=('Profit', 'sum'),
        Avg_Discount=('Discount', 'mean'),
        Units=('Quantity', 'sum')
    )
)

sub_agg['Unit_Price'] = sub_agg['Total_Sales'] / sub_agg['Units']
sub_agg['Margin_Per_Unit'] = sub_agg['Total_Profit'] / sub_agg['Units']

In [ ]:
# This code selects key performance features for each Sub-Category and scales them
# before fitting a 3-cluster K-means model.  
# Each Sub-Category is then assigned to a cluster based on its overall commercial profile.
sub_features = [
    'Total_Sales',
    'Total_Profit',
    'Avg_Discount',
    'Units',
    'Margin_Per_Unit'
]

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X_sub = sub_agg[sub_features].copy()

scaler_sub = StandardScaler()
X_sub_scaled = scaler_sub.fit_transform(X_sub)

kmeans_sub = KMeans(n_clusters=3, random_state=42, n_init=10)
sub_agg['Cluster'] = kmeans_sub.fit_predict(X_sub_scaled)

In [ ]:
# This block profiles each cluster by computing feature averages and counts,
# then assigns meaningful business labels (e.g., High Discount & High Loss)
# based on ranked discount and profit behaviour within each cluster.
sub_profile = (
    sub_agg
    .groupby('Cluster')[sub_features]
    .mean()
    .round(2)
)

sub_profile['Count'] = sub_agg.groupby('Cluster').size()

# Same labelling logic: high discount + low profit, high sales + healthy profit, rest = low risk
summary = sub_profile.copy()

high_disc_loss_idx = (
    summary
    .assign(
        disc_rank=summary['Avg_Discount'].rank(ascending=False),
        profit_rank=summary['Total_Profit'].rank(ascending=True)
    )
    .sort_values(['disc_rank', 'profit_rank'])
    .index[0]
)

high_sales_idx = summary['Total_Sales'].idxmax()

labels_sub = {}
labels_sub[high_disc_loss_idx] = 'High Discount – High Loss'
labels_sub[high_sales_idx] = 'High Sales – Healthy Profit'

for idx in summary.index:
    if idx not in labels_sub:
        labels_sub[idx] = 'Low Sales – Low Risk'

sub_agg['Cluster_Label'] = sub_agg['Cluster'].map(labels_sub)
sub_profile['Label'] = sub_profile.index.map(labels_sub)

sub_profile

In [ ]:
# This plot visualises sub-category-level profitability by displaying total profit 
# grouped by the K-means cluster labels. It highlights which sub-categories fall into
# high-loss, low-risk, or high-profit groups for clearer managerial insight.
px.bar(
    sub_agg.reset_index(),
    x='Sub-Category',
    y='Total_Profit',
    color='Cluster_Label',
    title='Sub-Category Clusters by Profit, Sales and Discount',
    barmode='group'
)